aQGC Matrix for reweighting xsec and uncertainties

This notebook reads cross-section and uncertainty data for aQGC operators and builds annotated 2D matrices for different operator groups (FM, FS, FT).

In [1]:
!pip install pandas numpy matplotlib seaborn --quiet

In [2]:
import pandas as pd
import numpy as np
import os
import re
import itertools
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
def parse_file(filepath):
    data = {}
    with open(filepath, "r") as f:
        for line in f:
            match = re.match(r"(.*?):\s*(-?\d*\.?\d+(?:[eE][-+]?\d+)?)", line.strip())
            if match:
                key, value = match.group(1), float(match.group(2))
                data[key] = value
    return data

#  Build 2D cross-section matrix (lower triangle only)
def build_matrix(xsec, op_list, process_name="WpZ_llqq"):
    mat = pd.DataFrame(index=op_list, columns=op_list, dtype=float)
    for i, op1 in enumerate(op_list):
        for j, op2 in enumerate(op_list):
            if i == j:
                key = f"{process_name}_{op1}_QUAD"
                mat.loc[op1, op2] = xsec.get(key, np.nan)
            elif i > j:
                key = f"{process_name}_{op2}vs{op1}_CROSS"
                mat.loc[op1, op2] = xsec.get(key, np.nan)
            else:
                mat.loc[op1, op2] = np.nan
    return mat

#  Plot heatmap of the matrix with scientific notation (2 significant figures)
def plot_matrix(matrix, outname, title, figsize=(12,8), cmap="YlGnBu", vmin=None, vmax=None):
    plt.figure(figsize=figsize)

    def cell_fmt(x):
        if np.isnan(x):
            return ""
        elif "Rel." in title and "%" in title:
            return f"{x:.1f}%"
        else:
            return f"{x:.2e}"

    annot = matrix.applymap(cell_fmt)
    sns.heatmap(matrix, annot=annot, fmt="", cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=0.5, square=True, cbar_kws={"label": title})
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outname + ".png")
    plt.savefig(outname + ".pdf")
    plt.show()
    plt.close()

In [10]:
# Uncertainty matrix (%)
def build_uncertainty_matrix(xsec, unc, op_list, process_name="WpZ_llqq"):
    mat = pd.DataFrame(index=op_list, columns=op_list, dtype=float)
    for i, op1 in enumerate(op_list):
        for j, op2 in enumerate(op_list):
            if i == j:
                key = f"{process_name}_{op1}_QUAD"
            elif i > j:
                key = f"{process_name}_{op2}vs{op1}_CROSS"
            else:
                continue
            u = unc.get(key, np.nan)
            mat.loc[op1, op2] =  u 
    return mat


# Relative uncertainty matrix (%)
def build_relative_uncertainty_matrix(xsec, unc, op_list, process_name="WpZ_llqq"):
    mat = pd.DataFrame(index=op_list, columns=op_list, dtype=float)
    for i, op1 in enumerate(op_list):
        for j, op2 in enumerate(op_list):
            if i == j:
                key = f"{process_name}_{op1}_QUAD"
            elif i > j:
                key = f"{process_name}_{op2}vs{op1}_CROSS"
            else:
                continue
            x = xsec.get(key, np.nan)
            u = unc.get(key, np.nan)
            mat.loc[op1, op2] = 100 * np.abs(u / x) if x != 0 else np.nan
    return mat

# Relative difference (%) between two matrices
def matrix_relative_diff(mat_ref, mat_cmp):
    return 100 * np.abs(mat_cmp - mat_ref) / np.abs(mat_ref)

# Pull significance map
def matrix_pull(mat_ref, unc_ref, mat_cmp, unc_cmp):
    return np.abs(mat_cmp - mat_ref) / np.sqrt(unc_ref**2 + unc_cmp**2)

In [ ]:
## User configuration

all_ops_cat = [
    "FM0", "FM1", "FM2", "FM3", "FM4", "FM5", "FM7", "FM8", "FM9", 
    "FS0", "FS1", "FS2",
    "FT0", "FT1", "FT2", "FT3", "FT4", "FT5", "FT6"
]
cross_terms = [f"{op1}vs{op2}" for op1, op2 in itertools.combinations(all_ops_cat, 2) if op1[:2] == op2[:2]]
Type_MC_all = [
    "EFTDec_Madspin", "EFTDec_polarisation",
    "Rwg_pol_50k", "Rwg_pol_100k", 
    "Rwg_InvSqrtXsec_50k", "Rwg_InvXsec_50k"
]
#Type_MC_all = ["EFTDec_Madspin","Rwg_pol_50k"]
groups = ["FM", "FS", "FT"]
#groups = ["FS"]

path_base = "/exp/atlas/salin/ATLAS/VBS_mc/plotting/Plot_Reweighting/Tables/Tables/Uncertainty/Xsec"
process_name = "WpZ_llqq"
origin = "MG"

In [41]:
path_base = "/exp/atlas/salin/ATLAS/VBS_mc/plotting/Plot_Reweighting/Tables/Tables/Uncertainty/Xsec"
process_name = "WpZ_llqq"
origin="MG"
Dir_for_output = f"./Matrix_plots_MG/{process_name}/"

def plot_matrix(matrix, outname, title, figsize=(12,8),fontsize=14, cmap="YlGnBu", vmin=None, vmax=None):
    plt.figure(figsize=figsize)

    def cell_fmt(x):
        if np.isnan(x):
            return ""
        elif "Rel." in title and "%" in title:
            return f"{x:.1f}%"
        else:
            return f"{x:.2e}"
    if "FM" in title:
        fontsize = 12
    elif "FS" in title:
        fontsize = 15
    elif "FT" in title:
        fontsize = 12
        
    annot = matrix.applymap(cell_fmt)
    sns.heatmap(
        matrix,
        annot=annot,
        fmt="",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        linewidths=0.5,
        square=True,
        cbar_kws={"label": title},
        annot_kws={"size": fontsize}
    )
    plt.title(title, fontsize=fontsize+4)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    plt.tight_layout()
    plt.savefig(outname + ".png")
    plt.savefig(outname + ".pdf")
    #plt.show()
    plt.close()

matrices = {}
uncertainties = {}
Rel_uncertainties = {}

for type_mc in Type_MC_all:
    lower_type_mc = type_mc.lower()
    file_xsec = f"{path_base}/VBS_cross_section_run2_{lower_type_mc}.txt"
    file_unc = f"{path_base}/VBS_Uncertainty_cross_section_run2_{lower_type_mc}.txt"

    if not os.path.exists(file_xsec) or not os.path.exists(file_unc):
        print(f" Missing files for: {type_mc}")
        continue

    print(f"Processing: {type_mc}")
    data_xsec = parse_file(file_xsec)
    data_unc = parse_file(file_unc)

    for group in groups:
        ops = [op for op in all_ops_cat if op.startswith(group)]
        matrix = build_matrix(data_xsec, ops)
        unc_matrix = build_uncertainty_matrix(data_xsec, data_unc, ops)
        Rel_unc_matrix = build_relative_uncertainty_matrix(data_xsec, data_unc, ops)

        outdir = f"./{Dir_for_output}/{type_mc}/"
        os.makedirs(outdir, exist_ok=True)

        #matrix.to_csv(outdir + f"matrix_{group}.csv")
        #unc_matrix.to_csv(outdir + f"uncertainty_matrix_{group}.csv")
        #Rel_unc_matrix.to_csv(outdir + f"Rel_uncertainty_matrix_{group}.csv")

        plot_matrix(matrix, outdir + f"matrix_{group}", f"From {origin}: {group} Cross Section [{type_mc}]")
        plot_matrix(unc_matrix, outdir + f"uncertainty_matrix_{group}", f"From {origin}: {group} Uncertainty on xsec [fb] [{type_mc}]", cmap="YlGnBu")
        plot_matrix(Rel_unc_matrix, outdir + f"Rel_uncertainty_matrix_{group}", f"From {origin}: {group} Rel. Uncertainty on xsec [%] [{type_mc}]", cmap="OrRd", vmin=0, vmax=30)

        matrices[(type_mc, group)] = matrix
        uncertainties[(type_mc, group)] = unc_matrix
        Rel_uncertainties[(type_mc, group)] = Rel_unc_matrix

# ==================== COMPARISON PLOTS ====================
REF= "EFTDec_Madspin"
Comp = ["Rwg_pol_50k", "Rwg_pol_100k", "Rwg_InvSqrtXsec_50k", "Rwg_InvXsec_50k"]
for COMP in Comp:
    for group in groups:
        m_ref = matrices.get((REF, group))
        m_cmp = matrices.get((COMP, group))
        u_ref = uncertainties.get((REF, group))
        u_cmp = uncertainties.get((COMP, group))

        if m_ref is None or m_cmp is None:
            continue

        diff = matrix_relative_diff(m_ref, m_cmp)
        pull = matrix_pull(m_ref, u_ref, m_cmp, u_cmp)

        outdir = f"./{Dir_for_output}/comparison_{REF}_vs_{COMP}/"
        os.makedirs(outdir, exist_ok=True)

        #diff.to_csv(outdir + f"rel_diff_{group}.csv")
        #pull.to_csv(outdir + f"pull_{group}.csv")

        plot_matrix(
            diff,
            outdir + f"rel_diff_{group}",f"From {origin}: {group} Rel. Diff" +
            r"$\left| \sigma_{\mathrm{EFTDec}} - \sigma_{\mathrm{Rwg}} \right| / \sigma_{\mathrm{EFTDec}}$ [in %]",
            cmap="OrRd",
            vmin=0,
            vmax=30
        )
        plot_matrix(pull, outdir + f"pull_{group}", f"From {origin}: {group} Pull {COMP} vs {REF}", cmap="OrRd")

Processing: EFTDec_Madspin
Processing: EFTDec_polarisation
Processing: Rwg_pol_50k
Processing: Rwg_pol_100k
Processing: Rwg_InvSqrtXsec_50k
Processing: Rwg_InvXsec_50k


In [37]:
### From Plot 

## User configuration

all_ops_cat = [
    "FM0", "FM1", "FM2", "FM3", "FM4", "FM5", "FM7", "FM8", "FM9", 
    "FS0", "FS1", "FS2",
    "FT0", "FT1", "FT2", "FT3", "FT4", "FT5", "FT6"
]
#all_ops_cat = ["FM0","FM1"]
cross_terms = [f"{op1}vs{op2}" for op1, op2 in itertools.combinations(all_ops_cat, 2) if op1[:2] == op2[:2]]
Type_MC_all = [
    "EFTDec_Madspin", "EFTDec_polarisation",
    "Rwg_pol_50k", "Rwg_pol_100k", 
    "Rwg_InvSqrtXsec_50k", "Rwg_InvXsec_50k"
]
#Type_MC_all = ["EFTDec_Madspin","Rwg_pol_50k"]
groups = ["FM", "FS", "FT"]
#groups = ["FS"]

path_base = "/exp/atlas/salin/ATLAS/VBS_mc/plotting/Plot_Reweighting/Tables/Tables/Uncertainty/Xsec/FromPlot/"
process_name = "WpZ_llqq"
origin = "Signal Region"



In [42]:

path_base = "/exp/atlas/salin/ATLAS/VBS_mc/plotting/Plot_Reweighting/Tables/Tables/Uncertainty/Xsec/FromPlot"
process_name = "WpZ_llqq"
origin="SR"
Dir_for_output = f"./Matrix_plots_SR/{process_name}/"

def plot_matrix(matrix, outname, title, figsize=(12,8), cmap="YlGnBu", vmin=None, vmax=None):
    plt.figure(figsize=figsize)

    def cell_fmt(x):
        if np.isnan(x):
            return ""
        elif "Rel." in title and "%" in title:
            return f"{x:.1f}%"
        else:
            return f"{x:.2e}"
    if "FM" in title:
        fontsize = 12
    elif "FS" in title:
        fontsize = 15
    elif "FT" in title:
        fontsize = 12
    annot = matrix.applymap(cell_fmt)
    sns.heatmap(matrix, annot=annot, fmt="", cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=0.5, square=True, cbar_kws={"label": title})
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outname + ".png")
    plt.savefig(outname + ".pdf")
    #plt.show()
    plt.close()

matrices = {}
uncertainties = {}
Rel_uncertainties = {}

for type_mc in Type_MC_all:
    lower_type_mc = type_mc.lower()
    
    file_xsec = f"{path_base}/VBS_cross_section_run2_{type_mc}.txt"
    file_unc = f"{path_base}/VBS_Uncertainty_cross_section_run2_{type_mc}.txt"
    print(f"Looking for files: {file_xsec} and {file_unc}")
    if not os.path.exists(file_xsec) or not os.path.exists(file_unc):
        print(f" Missing files for: {type_mc}")
        continue

    print(f"Processing: {type_mc}")
    data_xsec = parse_file(file_xsec)
    data_unc = parse_file(file_unc)

    for group in groups:
        ops = [op for op in all_ops_cat if op.startswith(group)]
        matrix = build_matrix(data_xsec, ops)
        unc_matrix = build_uncertainty_matrix(data_xsec, data_unc, ops)
        Rel_unc_matrix = build_relative_uncertainty_matrix(data_xsec, data_unc, ops)

        outdir = f"./{Dir_for_output}/{type_mc}/"
        os.makedirs(outdir, exist_ok=True)

        #matrix.to_csv(outdir + f"matrix_{group}.csv")
        #unc_matrix.to_csv(outdir + f"uncertainty_matrix_{group}.csv")
        #Rel_unc_matrix.to_csv(outdir + f"Rel_uncertainty_matrix_{group}.csv")

        plot_matrix(matrix, outdir + f"matrix_{group}", f"From {origin}: {group} Cross Section [{type_mc}]")
        plot_matrix(unc_matrix, outdir + f"uncertainty_matrix_{group}", f"From {origin}: {group} Uncertainty on xsec [fb] [{type_mc}]", cmap="YlGnBu")
        plot_matrix(Rel_unc_matrix, outdir + f"Rel_uncertainty_matrix_{group}", f"From {origin}: {group} Rel. Uncertainty on xsec [%] [{type_mc}]", cmap="OrRd", vmin=0, vmax=30)

        matrices[(type_mc, group)] = matrix
        uncertainties[(type_mc, group)] = unc_matrix
        Rel_uncertainties[(type_mc, group)] = Rel_unc_matrix

# ==================== COMPARISON PLOTS ====================
REF= "EFTDec_Madspin"
Comp = ["Rwg_pol_50k", "Rwg_pol_100k", "Rwg_InvSqrtXsec_50k", "Rwg_InvXsec_50k"]
for COMP in Comp:
    for group in groups:
        m_ref = matrices.get((REF, group))
        m_cmp = matrices.get((COMP, group))
        u_ref = uncertainties.get((REF, group))
        u_cmp = uncertainties.get((COMP, group))

        if m_ref is None or m_cmp is None:
            continue

        diff = matrix_relative_diff(m_ref, m_cmp)
        pull = matrix_pull(m_ref, u_ref, m_cmp, u_cmp)

        outdir = f"./{Dir_for_output}/comparison_{REF}_vs_{COMP}/"
        os.makedirs(outdir, exist_ok=True)

        #diff.to_csv(outdir + f"rel_diff_{group}.csv")
        #pull.to_csv(outdir + f"pull_{group}.csv")
        plot_matrix(
            diff,
            outdir + f"rel_diff_{group}", f"From {origin}: {group} Rel. Diff" +
            r"$\left| \sigma_{\mathrm{EFTDec}} - \sigma_{\mathrm{Rwg}} \right| / \sigma_{\mathrm{EFTDec}}$ [in %]",
            cmap="OrRd",
            vmin=0,
            vmax=30
        )
        plot_matrix(pull, outdir + f"pull_{group}", f"From {origin}: {group} Pull {COMP} vs {REF}", cmap="OrRd")

Looking for files: /exp/atlas/salin/ATLAS/VBS_mc/plotting/Plot_Reweighting/Tables/Tables/Uncertainty/Xsec/FromPlot/VBS_cross_section_run2_EFTDec_Madspin.txt and /exp/atlas/salin/ATLAS/VBS_mc/plotting/Plot_Reweighting/Tables/Tables/Uncertainty/Xsec/FromPlot/VBS_Uncertainty_cross_section_run2_EFTDec_Madspin.txt
Processing: EFTDec_Madspin
Looking for files: /exp/atlas/salin/ATLAS/VBS_mc/plotting/Plot_Reweighting/Tables/Tables/Uncertainty/Xsec/FromPlot/VBS_cross_section_run2_EFTDec_polarisation.txt and /exp/atlas/salin/ATLAS/VBS_mc/plotting/Plot_Reweighting/Tables/Tables/Uncertainty/Xsec/FromPlot/VBS_Uncertainty_cross_section_run2_EFTDec_polarisation.txt
 Missing files for: EFTDec_polarisation
Looking for files: /exp/atlas/salin/ATLAS/VBS_mc/plotting/Plot_Reweighting/Tables/Tables/Uncertainty/Xsec/FromPlot/VBS_cross_section_run2_Rwg_pol_50k.txt and /exp/atlas/salin/ATLAS/VBS_mc/plotting/Plot_Reweighting/Tables/Tables/Uncertainty/Xsec/FromPlot/VBS_Uncertainty_cross_section_run2_Rwg_pol_50k.